# Use Case 3 — Build EPC Causal TKG

**Goal:** Construct the Temporal Knowledge Graph from the four data sources and import into Neo4j.

**Quadruple format:** `(subject, relation, object, valid_from, valid_to)`

**Relations:**
- `DELAYED_BY_EVENT`: Activity <- Event (NCR, ChangeOrder)
- `LINKED_PO`: Activity <- PurchaseOrder
- `REQUIRES_DOC`: Activity <- Document
- `DEPENDS_ON`: Activity <- Activity (predecessor)
- `DELIVERED_LATE`: PurchaseOrder (state at delivery time)
- `APPROVED_LATE`: Document (state at approval time)


## 0. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/UseCase3')

df_act  = pd.read_csv(DATA_DIR / 'activities.csv', parse_dates=['planned_start','planned_finish','actual_start','actual_finish'])
df_proc = pd.read_csv(DATA_DIR / 'procurement.csv', parse_dates=['planned_order_dt','planned_deliv_dt','actual_deliv_dt'])
df_docs = pd.read_csv(DATA_DIR / 'documents.csv', parse_dates=['issue_date','planned_appr_date','actual_appr_date'])
df_ev   = pd.read_csv(DATA_DIR / 'events.csv', parse_dates=['date'])
with open(DATA_DIR / 'causal_ground_truth.json') as f:
    gt = json.load(f)

print('Data loaded OK')


## 1. Build Quadruples


In [ ]:
quads = []  # (subject, relation, object, t_start, t_end, properties)
TX_NOW = datetime.now(timezone.utc).isoformat()

# Activity nodes + planned/actual timing
for _, row in df_act.iterrows():
    aid = row['activity_id']
    # BELONGS_TO_WP
    quads.append((aid, 'BELONGS_TO_WP', row['work_package'],
                  row['planned_start'], row['planned_finish'], {}))
    # DELAYED (state fact)
    if row['delayed']:
        quads.append((aid, 'IS_DELAYED', 'TRUE',
                      row['actual_start'] if pd.notna(row['actual_start']) else row['planned_start'],
                      row['actual_finish'] if pd.notna(row['actual_finish']) else row['planned_finish'],
                      {'reason': str(row.get('delay_reason', ''))}))

# Event -> Activity impacts
for _, row in df_ev.iterrows():
    if pd.notna(row['impacts_activity']):
        quads.append((row['event_id'], 'IMPACTS_ACTIVITY', row['impacts_activity'],
                      row['date'], row['date'],
                      {'event_type': row['event_type'],
                       'delay_days': row.get('triggers_delay_days', 0)}))

# PO -> Activity links + delivery state
for _, row in df_proc.iterrows():
    quads.append((row['po_id'], 'LINKED_TO_ACTIVITY', row['linked_activity'],
                  row['planned_order_dt'], row['planned_deliv_dt'], {}))
    if row.get('delay_days', 0) > 0:
        deliv = row['actual_deliv_dt'] if pd.notna(row['actual_deliv_dt']) else row['planned_deliv_dt']
        quads.append((row['po_id'], 'DELIVERED_LATE', row['linked_activity'],
                      row['planned_deliv_dt'], deliv,
                      {'delay_days': row['delay_days']}))

# Document -> Activity links + late approval
for _, row in df_docs.iterrows():
    quads.append((row['doc_id'], 'LINKED_TO_ACTIVITY', row['linked_activity'],
                  row['issue_date'], row['planned_appr_date'], {}))
    if row.get('late_days', 0) > 0:
        quads.append((row['doc_id'], 'APPROVED_LATE', row['linked_activity'],
                      row['planned_appr_date'],
                      row['actual_appr_date'] if pd.notna(row['actual_appr_date']) else row['planned_appr_date'],
                      {'late_days': row['late_days']}))

df_quads = pd.DataFrame(quads, columns=['subject','relation','object','t_start','t_end','props'])
print(f'Total quadruples: {len(df_quads)}')
print(df_quads.groupby('relation').size().to_string())


## 2. Graph Statistics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Relation distribution
rel_counts = df_quads.groupby('relation').size().sort_values(ascending=True)
rel_counts.plot.barh(ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Quadruples by Relation Type')
axes[0].set_xlabel('Count')

# Timeline of quadruples
df_q_ts = df_quads.dropna(subset=['t_start'])
df_q_ts['month'] = pd.to_datetime(df_q_ts['t_start']).dt.to_period('M')
monthly = df_q_ts.groupby(['month','relation']).size().unstack(fill_value=0)
monthly.plot.bar(ax=axes[1], stacked=True, alpha=0.8)
axes[1].set_title('Quadruples by Month')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

# Entity counts
all_subjects = set(df_quads['subject'].unique())
all_objects  = set(df_quads['object'].unique())
all_entities = all_subjects | all_objects
print(f'Total unique entities: {len(all_entities)}')
print(f'  Activities: {len([e for e in all_entities if e.startswith("A")])}')
print(f'  Events:     {len([e for e in all_entities if e.startswith(("CO","NCR","INS","WD","DN"))])}')
print(f'  POs:        {len([e for e in all_entities if e.startswith("PO")])}')
print(f'  Documents:  {len([e for e in all_entities if e.startswith("DOC")])}')


## 3. Neo4j Import (optional)
> Requires Neo4j at bolt://172.22.43.151:7687


In [ ]:
NEO4J_URI  = 'bolt://172.22.43.151:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASS = 'your_password'

try:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    driver.verify_connectivity()
    NEO4J_OK = True
    print('Neo4j connected')
except Exception as e:
    NEO4J_OK = False
    print(f'Neo4j not available: {e}')


In [ ]:
if NEO4J_OK:
    with driver.session() as s:
        s.run('MATCH (n) WHERE n:Activity OR n:PurchaseOrder OR n:Document OR n:Event DETACH DELETE n')

        for _, row in df_act.iterrows():
            s.run('''
                MERGE (a:Activity {id: $id})
                SET a.work_package=$wp, a.discipline=$disc,
                    a.planned_start=$ps, a.planned_finish=$pf,
                    a.actual_start=$as_, a.actual_finish=$af,
                    a.delayed=$delayed, a.tx_time=$tx
            ''', id=row['activity_id'], wp=row['work_package'],
                 disc=row['discipline'],
                 ps=str(row['planned_start'])[:10], pf=str(row['planned_finish'])[:10],
                 as_=str(row['actual_start'])[:10] if pd.notna(row['actual_start']) else None,
                 af=str(row['actual_finish'])[:10] if pd.notna(row['actual_finish']) else None,
                 delayed=bool(row['delayed']), tx=TX_NOW)

        for _, row in df_ev.iterrows():
            s.run('''
                MERGE (e:Event {id: $id})
                SET e.event_type=$etype, e.date=$date, e.tx_time=$tx
                WITH e
                MATCH (a:Activity {id: $act_id})
                MERGE (e)-[r:IMPACTS_ACTIVITY]->(a)
                SET r.delay_days=$dd, r.date=$date
            ''', id=row['event_id'], etype=row['event_type'],
                 date=str(row['date'])[:10],
                 act_id=row['impacts_activity'],
                 dd=float(row['triggers_delay_days']) if pd.notna(row['triggers_delay_days']) else 0,
                 tx=TX_NOW)

        for _, row in df_proc.iterrows():
            s.run('''
                MERGE (po:PurchaseOrder {id: $id})
                SET po.vendor=$vendor, po.delay_days=$dd,
                    po.delivery_status=$status, po.tx_time=$tx
                WITH po
                MATCH (a:Activity {id: $act_id})
                MERGE (po)-[:LINKED_TO_ACTIVITY]->(a)
            ''', id=row['po_id'], vendor=row['vendor'],
                 dd=float(row['delay_days']) if pd.notna(row['delay_days']) else 0,
                 status=row['delivery_status'], act_id=row['linked_activity'], tx=TX_NOW)

        r = driver.session().run('MATCH (n) RETURN labels(n)[0] AS l, count(n) AS c')
        for rec in r:
            print(f'  {rec["l"]:<20}: {rec["c"]}')
    driver.close()
else:
    print('Skipping import. TKG built in-memory (df_quads).')
    print(f'Quadruples ready: {len(df_quads)}')


## Summary

| Relation | Count | Temporal |
|----------|-------|----------|
| BELONGS_TO_WP | 60 | planned_start - planned_finish |
| IS_DELAYED | 8 | actual_start - actual_finish |
| IMPACTS_ACTIVITY | 24 | event date (point) |
| LINKED_TO_ACTIVITY | 48 | order_dt - delivery_dt |
| DELIVERED_LATE | varies | planned_deliv - actual_deliv |
| APPROVED_LATE | varies | planned_appr - actual_appr |

**Next:** `03_tlogic_causal.ipynb` — T-Logic rule validation against causal ground truth
